# Step 3 — UMLS linking + Llama 3.1 disambiguation

Use this notebook on **Google Colab** (GPU runtime recommended for Llama).

**Defaults** target `outputs/pipeline-output18/step2` (Step 2 output; same as local `pipeline/step3/`).

1. Upload your thesis repo (ZIP) or `git clone` so `pipeline/` and `input/UMLS.csv` exist.
2. Ensure Step 2 has produced CSVs under `outputs/pipeline-output18/step2/`.
3. Set `HF_TOKEN` (Colab Secrets or input) for gated **Llama 3.1** on the Hub.

In [ ]:
# Install stack for local Llama (Colab GPU)
!pip install -q torch transformers accelerate sentencepiece rapidfuzz PyMuPDF

In [ ]:
import os, sys, zipfile
from pathlib import Path

CONTENT = Path("/content")

# Optional: os.environ["COLAB_THESIS_ROOT"] = "/content/MyFolder"


def find_repo_root() -> Path | None:
    env = os.environ.get("COLAB_THESIS_ROOT", "").strip()
    if env:
        p = Path(env).expanduser().resolve()
        if (p / "pipeline").is_dir():
            return p
    if (CONTENT / "pipeline").is_dir():
        return CONTENT.resolve()
    for c in sorted(CONTENT.iterdir(), key=lambda x: x.name.lower()):
        if c.is_dir() and (c / "pipeline").is_dir():
            return c.resolve()
    return None


UPLOAD_ZIP = True  # False if you already cloned/unzipped under /content

if UPLOAD_ZIP:
    from google.colab import files

    print("Upload thesis ZIP (root must contain pipeline/)")
    uploaded = files.upload()
    for name, data in uploaded.items():
        p = CONTENT / name
        p.write_bytes(data)
        if name.lower().endswith(".zip"):
            with zipfile.ZipFile(p, "r") as zf:
                zf.extractall(str(CONTENT))
        break

ROOT = find_repo_root()
if ROOT is None:
    raise FileNotFoundError(
        "No folder under /content contains pipeline/. "
        "Unzip at /content or set COLAB_THESIS_ROOT."
    )

os.chdir(ROOT)
sys.path.insert(0, str(ROOT))
print("ROOT =", ROOT)

In [ ]:
# Hugging Face token for meta-llama/Llama-3.1-8B-Instruct (gated)
import os

try:
    from google.colab import userdata
    HF_TOKEN = userdata.get('HF_TOKEN')
except Exception:
    HF_TOKEN = os.environ.get('HF_TOKEN')

if not HF_TOKEN:
    HF_TOKEN = input('Paste HF_TOKEN (read access): ').strip()
os.environ['HF_TOKEN'] = HF_TOKEN
print('HF_TOKEN set:', bool(HF_TOKEN))

In [ ]:
# 3a — UMLS linking (symbolic + fuzzy); writes grounded_entities.json
from pipeline.step3.entity_link_sources import run_entity_linking

SHAPED = ROOT / "outputs" / "pipeline-output18" / "step2"
UMLS = ROOT / 'input' / 'UMLS.csv'

report = run_entity_linking(SHAPED, UMLS, write_jsonl=True)
report

In [ ]:
# 3b — Llama 3.1 disambiguation (only rows with needs_disambiguation)
from pipeline.step3.disambiguate_llama import disambiguate_llama

GROUNDED = SHAPED / 'grounded_entities.json'
out = disambiguate_llama(GROUNDED, SHAPED, hf_model='meta-llama/Llama-3.1-8B-Instruct', batch_size=4)
out